In [0]:
# ===================================================
# BLOCK 1 — IMPORTS AND CONFIGURATION
# ===================================================

from datetime import datetime, timezone

from pyspark.sql import functions as F


"""
Validate the persisted Silver production-lot and quarantine tables.

The controls verify typing, uniqueness, referential integrity, quantity
reconciliation, yield accuracy, quarantine reasons, and end-to-end row
reconciliation against Bronze.
"""

BRONZE_TABLE = "semiconplus_portfolio.bronze.production_lots"
SILVER_TABLE = "semiconplus_portfolio.silver.production_lots"
QUARANTINE_TABLE = "semiconplus_portfolio.quarantine.production_lots"

VALIDATION_TIME_UTC = datetime.now(timezone.utc)

print(f"Validation time UTC: {VALIDATION_TIME_UTC.isoformat()}")

In [0]:
# ===================================================
# BLOCK 2 — TABLE AVAILABILITY
# ===================================================

"""
Confirm that all required pipeline tables are available before running
cross-layer reconciliation.
"""

required_tables = [BRONZE_TABLE, SILVER_TABLE, QUARANTINE_TABLE]

missing_tables = [
    table_name
    for table_name in required_tables
    if not spark.catalog.tableExists(table_name)
]

assert not missing_tables, f"Required tables are missing: {missing_tables}"

bronze_df = spark.table(BRONZE_TABLE)
silver_df = spark.table(SILVER_TABLE)
quarantine_df = spark.table(QUARANTINE_TABLE)

print("Required validation tables are available.")

In [0]:
# ===================================================
# BLOCK 3 — CROSS-LAYER RECONCILIATION
# ===================================================

"""
Reconcile all accepted and rejected records to the complete Bronze
source population.
"""

bronze_count = bronze_df.count()
silver_count = silver_df.count()
quarantine_count = quarantine_df.count()

print(f"Bronze rows: {bronze_count:,}")
print(f"Silver rows: {silver_count:,}")
print(f"Quarantine rows: {quarantine_count:,}")

assert silver_count + quarantine_count == bronze_count, (
    "Cross-layer reconciliation failed."
)

assert bronze_count == 18_126

print("Cross-layer reconciliation passed.")

In [0]:
# ===================================================
# BLOCK 4 — SILVER UNIQUENESS
# ===================================================

"""
Confirm that each accepted production-lot business key occurs exactly
once in the Silver table.
"""

duplicate_silver_keys_df = (
    silver_df
    .groupBy("lot_id")
    .count()
    .filter(F.col("count") > 1)
)

duplicate_silver_key_count = duplicate_silver_keys_df.count()

display(duplicate_silver_keys_df)

assert duplicate_silver_key_count == 0, (
    f"Silver contains {duplicate_silver_key_count} duplicate lot IDs."
)

print("Silver uniqueness validation passed.")

In [0]:
# ===================================================
# BLOCK 5 — REQUIRED-FIELD VALIDATION
# ===================================================

"""
Confirm that accepted Silver records contain all required business and
lineage fields.
"""

required_columns = [
    "lot_id",
    "production_date",
    "device_id",
    "product_group_id",
    "site_id",
    "equipment_id",
    "start_timestamp_utc",
    "quantity_started",
    "quantity_passed",
    "quantity_failed",
    "calculated_yield",
    "_source_file_path",
    "_bronze_pipeline_run_id",
    "_silver_pipeline_run_id",
    "_silver_processed_at_utc",
]

required_field_failures = (
    silver_df
    .select(
        *[
            F.sum(
                F.when(F.col(column_name).isNull(), 1).otherwise(0)
            ).alias(f"null_{column_name.lstrip('_')}")
            for column_name in required_columns
        ]
    )
    .first()
    .asDict()
)

for metric_name, metric_value in required_field_failures.items():
    print(f"{metric_name}: {metric_value}")
    assert metric_value == 0, (
        f"Required-field validation failed: {metric_name}={metric_value}"
    )

print("Required-field validation passed.")

In [0]:
# ===================================================
# BLOCK 6 — QUANTITY AND YIELD VALIDATION
# ===================================================

"""
Verify quantity reconciliation and calculated-yield accuracy for every
accepted Silver record.
"""

quantity_failure_count = silver_df.filter(
    (F.col("quantity_started") <= 0)
    | (F.col("quantity_passed") < 0)
    | (F.col("quantity_failed") < 0)
    | (
        F.col("quantity_passed") + F.col("quantity_failed")
        != F.col("quantity_started")
    )
).count()

yield_failure_count = silver_df.filter(
    (F.col("calculated_yield") < F.lit(0))
    | (F.col("calculated_yield") > F.lit(1))
    | (
        F.abs(
            F.col("calculated_yield").cast("double")
            - (
                F.col("quantity_passed") / F.col("quantity_started")
            )
        ) > F.lit(0.000001)
    )
).count()

print(f"Quantity failures: {quantity_failure_count}")
print(f"Calculated-yield failures: {yield_failure_count}")

assert quantity_failure_count == 0
assert yield_failure_count == 0

print("Quantity and yield validation passed.")

In [0]:
# ===================================================
# BLOCK 7 — DEVICE-REFERENCE VALIDATION
# ===================================================

"""
Confirm that accepted Silver records contain no unresolved device
identifiers and that the deliberate unknown-device record was quarantined.
"""

DEVICE_REFERENCE_PATH = (
    "/Volumes/semiconplus_portfolio/"
    "landing/external_source/reference/devices.csv"
)

valid_devices_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .load(DEVICE_REFERENCE_PATH)
    .select(F.trim(F.col("device_id")).alias("reference_device_id"))
    .dropDuplicates(["reference_device_id"])
)

unresolved_silver_devices = (
    silver_df
    .join(
        F.broadcast(valid_devices_df),
        silver_df.device_id == valid_devices_df.reference_device_id,
        "left_anti",
    )
    .count()
)

unknown_device_quarantine_count = quarantine_df.filter(
    F.array_contains(
        F.col("_quality_reasons"),
        "UNKNOWN_DEVICE_ID",
    )
).count()

print(f"Unresolved Silver device IDs: {unresolved_silver_devices}")
print(
    "Unknown-device quarantine records: "
    f"{unknown_device_quarantine_count}"
)

assert unresolved_silver_devices == 0
assert unknown_device_quarantine_count == 1

print("Device-reference validation passed.")

In [0]:
# ===================================================
# BLOCK 8 — QUARANTINE REASON VALIDATION
# ===================================================

"""
Confirm that every quarantined record contains at least one rejection
reason and summarize the rejected population by quality rule.
"""

missing_reason_count = quarantine_df.filter(
    F.col("_quality_reasons").isNull()
    | (F.size(F.col("_quality_reasons")) == 0)
).count()

quarantine_reason_summary_df = (
    quarantine_df
    .select(F.explode("_quality_reasons").alias("quality_reason"))
    .groupBy("quality_reason")
    .count()
    .orderBy("quality_reason")
)

display(quarantine_reason_summary_df)

assert missing_reason_count == 0

assert quarantine_df.filter(
    F.array_contains(F.col("_quality_reasons"), "DUPLICATE_LOT_ID")
).count() == 1

assert quarantine_df.filter(
    F.array_contains(F.col("_quality_reasons"), "UNKNOWN_DEVICE_ID")
).count() == 1

print("Quarantine-reason validation passed.")

In [0]:
# ===================================================
# BLOCK 9 — DATA-TYPE VALIDATION
# ===================================================

"""
Confirm that persisted Silver columns use the approved analytical data
types required by downstream aggregations and data marts.
"""

actual_types = dict(silver_df.dtypes)

expected_types = {
    "production_date": "date",
    "start_timestamp_utc": "timestamp",
    "quantity_started": "bigint",
    "quantity_passed": "bigint",
    "quantity_failed": "bigint",
    "source_actual_yield": "decimal(9,6)",
    "calculated_yield": "decimal(9,6)",
}

for column_name, expected_type in expected_types.items():
    actual_type = actual_types.get(column_name)
    print(f"{column_name}: {actual_type}")
    assert actual_type == expected_type, (
        f"Expected {column_name} to use {expected_type}, "
        f"found {actual_type}."
    )

print("Silver data-type validation passed.")

In [0]:
# ===================================================
# BLOCK 10 — DATE-COVERAGE VALIDATION
# ===================================================

"""
Confirm that the accepted Silver population retains the expected
historical production-date coverage.
"""

date_metrics = (
    silver_df
    .agg(
        F.min("production_date").alias("minimum_production_date"),
        F.max("production_date").alias("maximum_production_date"),
    )
    .first()
    .asDict()
)

print(date_metrics)

assert str(date_metrics["minimum_production_date"]) == "2021-01-01"
assert str(date_metrics["maximum_production_date"]) == "2025-12-31"

print("Silver date-coverage validation passed.")

In [0]:
# ===================================================
# BLOCK 11 — RERUN IDEMPOTENCY BASELINE
# ===================================================

"""
Capture the current output counts before repeating the deterministic
Silver transformation.
"""

silver_count_before_rerun = spark.table(SILVER_TABLE).count()
quarantine_count_before_rerun = spark.table(QUARANTINE_TABLE).count()

print(f"Silver count before rerun: {silver_count_before_rerun:,}")
print(
    "Quarantine count before rerun: "
    f"{quarantine_count_before_rerun:,}"
)

In [0]:
# ===================================================
# BLOCK 12 — RERUN IDEMPOTENCY VALIDATION
# ===================================================

"""
Confirm that repeating the Silver transformation replaces the validated
snapshots without changing row counts or creating duplicate records.
"""

silver_count_after_rerun = spark.table(SILVER_TABLE).count()
quarantine_count_after_rerun = spark.table(QUARANTINE_TABLE).count()

print(f"Silver count after rerun: {silver_count_after_rerun:,}")
print(
    "Quarantine count after rerun: "
    f"{quarantine_count_after_rerun:,}"
)

assert silver_count_after_rerun == silver_count_before_rerun
assert quarantine_count_after_rerun == quarantine_count_before_rerun

assert (
    spark.table(SILVER_TABLE)
    .groupBy("lot_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
    == 0
)

print("SILVER RERUN IDEMPOTENCY TEST PASSED")

In [0]:
# ===================================================
# BLOCK 13 — FINAL VALIDATION SUMMARY
# ===================================================

"""
Publish the final Silver validation result for execution evidence and
project documentation.
"""

final_result = {
    "status": "PASSED",
    "validated_at_utc": VALIDATION_TIME_UTC.isoformat(),
    "bronze_rows": bronze_count,
    "silver_rows": silver_count,
    "quarantine_rows": quarantine_count,
    "silver_duplicate_keys": duplicate_silver_key_count,
    "quantity_failures": quantity_failure_count,
    "yield_failures": yield_failure_count,
    "unresolved_silver_devices": unresolved_silver_devices,
    "quarantine_records_without_reasons": missing_reason_count,
}

print("SILVER VALIDATION PASSED")

for metric_name, metric_value in final_result.items():
    print(f"{metric_name}: {metric_value}")


In [0]:
%sql
-- Review accepted and quarantined row counts.
SELECT 'bronze' AS layer, COUNT(*) AS row_count
FROM semiconplus_portfolio.bronze.production_lots
UNION ALL
SELECT 'silver', COUNT(*)
FROM semiconplus_portfolio.silver.production_lots
UNION ALL
SELECT 'quarantine', COUNT(*)
FROM semiconplus_portfolio.quarantine.production_lots;

In [0]:
%sql
-- Review quarantine reasons.
SELECT
    quality_reason,
    COUNT(*) AS rejected_record_count
FROM semiconplus_portfolio.quarantine.production_lots
LATERAL VIEW explode(_quality_reasons) reasons AS quality_reason
GROUP BY quality_reason
ORDER BY quality_reason;

In [0]:
%sql
-- Confirm Silver uniqueness.
SELECT
    lot_id,
    COUNT(*) AS record_count
FROM semiconplus_portfolio.silver.production_lots
GROUP BY lot_id
HAVING COUNT(*) > 1;
